# 03 — BAMIC wine multi-seed analysis (Dr. Cao, Point 1)

Reads the 10 `_ms1` run folders written by notebook 02 and quantifies
between-seed variability against within-fit posterior variability via the law
of total variance,
$\operatorname{Var}(Z_i) = \mathbb{E}\{\operatorname{Var}(Z_i\mid \text{seed})\} + \operatorname{Var}\{\mathbb{E}(Z_i\mid \text{seed})\}$,
on the document probability scale, the document logit scale, and the token
contribution scale (200-document panel), plus routing stability, a pooled
**seed-ensemble** predictive (a constructive output: pooled draws across seeds
approximate the total-uncertainty interval Dr. Cao is asking about), and a
side-by-side comparison with the numbers Codex reported from its independent
reimplementation.

Everything is written to `multiseed_results_ms1/` in Drive: CSVs, LaTeX tables,
macros, and `multiseed_claude_summary.json` (send that JSON plus the CSVs back
for the joint write-up). CPU runtime is fine for this notebook.

In [1]:

# ================================================================
# 0. Mount Drive and locate files. Notebooks stay on YOUR computer,
# same as your usual workflow: open this notebook in Colab from your
# local folder; only data/outputs live in Drive. If the official 01d
# notebook is needed and not already in this Colab session, a file
# picker opens so you can choose it from your local folder.
# If anything is guessed wrong, set NOTEBOOK_PATH and/or OUTPUTS_ROOT
# manually below and rerun this cell.
# ================================================================
import os
from pathlib import Path

REQUIRE_OFFICIAL_NB = False   # this notebook needs the 01d file itself

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
    print('[note] not on Colab — set paths manually below.')

MYDRIVE = Path('/content/drive/MyDrive')
NOTEBOOK_PATH = None   # manual override, e.g. '/content/01d_bamic_wine_gatefix_bestval.ipynb'
OUTPUTS_ROOT = None    # manual override, e.g. '/content/drive/MyDrive/AMIC project/wine_benchmark/outputs'

OFFICIAL_NB = '01d_bamic_wine_gatefix_bestval.ipynb'
_PRUNE = {'drive', 'sample_data', '.config'}

def _walk_find(match, roots, max_depth=5, want_dir=False):
    """Bounded filename search, skipping hidden dirs and the Drive mount."""
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        base = str(root).rstrip('/').count('/')
        for dirpath, dirnames, filenames in os.walk(root):
            if dirpath.count('/') - base >= max_depth:
                dirnames[:] = []
                continue
            dirnames[:] = [d for d in dirnames
                           if not d.startswith('.') and d not in _PRUNE]
            names = dirnames if want_dir else filenames
            for n in names:
                if match(n):
                    return Path(dirpath) / n
    return None

if REQUIRE_OFFICIAL_NB and NOTEBOOK_PATH is None:
    # 1) already in this Colab session (drag-dropped into the Files panel,
    #    or uploaded on a previous run of this cell)
    for _cand in [Path('/content') / OFFICIAL_NB, Path.cwd() / OFFICIAL_NB]:
        if _cand.exists():
            NOTEBOOK_PATH = _cand
            break
    if NOTEBOOK_PATH is None:
        NOTEBOOK_PATH = _walk_find(lambda n: n == OFFICIAL_NB,
                                   [Path('/content')], max_depth=3)
    # 2) a copy kept in Drive (optional)
    if NOTEBOOK_PATH is None:
        _guess = MYDRIVE / 'AMIC project' / 'wine_benchmark' / OFFICIAL_NB
        NOTEBOOK_PATH = _guess if _guess.exists() else _walk_find(
            lambda n: n == OFFICIAL_NB, [MYDRIVE / 'AMIC project', MYDRIVE])
    # 3) pick it from your local computer (your normal upload workflow)
    if NOTEBOOK_PATH is None and ON_COLAB:
        print(f'{OFFICIAL_NB} is not in this session or Drive.')
        print('Choose it from your local folder, e.g. '
              'BAMIC_code/dr_cao_multiseed/ or '
              'BAMIC_code/BAMIC_codex_updated/wine_benchmark/:')
        from google.colab import files
        _up = files.upload()
        for _name in _up:
            if '01d_bamic_wine_gatefix_bestval' in _name and _name.endswith('.ipynb'):
                NOTEBOOK_PATH = Path('/content') / _name

if REQUIRE_OFFICIAL_NB:
    assert NOTEBOOK_PATH is not None and Path(NOTEBOOK_PATH).exists(), (
        f'Could not obtain {OFFICIAL_NB}. Rerun this cell and choose the file '
        f'in the upload dialog, or set NOTEBOOK_PATH manually above.')
    NOTEBOOK_PATH = str(NOTEBOOK_PATH)
    print('NOTEBOOK_PATH:', NOTEBOOK_PATH)

if OUTPUTS_ROOT is None:
    _guess = MYDRIVE / 'AMIC project' / 'wine_benchmark' / 'outputs'
    if _guess.exists() and any(_guess.glob('wine_bamic_*')):
        OUTPUTS_ROOT = _guess
    else:
        _hit = _walk_find(lambda n: n.startswith('wine_bamic_'),
                          [MYDRIVE / 'AMIC project', MYDRIVE], want_dir=True)
        OUTPUTS_ROOT = _hit.parent if _hit is not None else _guess
OUTPUTS_ROOT = Path(OUTPUTS_ROOT)
OUTPUTS_ROOT.mkdir(parents=True, exist_ok=True)
print('OUTPUTS_ROOT :', OUTPUTS_ROOT)
_existing = sorted(p.name for p in OUTPUTS_ROOT.glob('wine_bamic_*'))
print(f'existing wine_bamic_* run folders here: {len(_existing)}')
for _n in _existing[:8]:
    print('  ', _n)
if not _existing:
    print('  [check] none found — if runs exist elsewhere, set OUTPUTS_ROOT '
          'manually above. (For a first-ever run this is fine.)')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OUTPUTS_ROOT : /content/drive/MyDrive/AMIC project/wine_benchmark/outputs
existing wine_bamic_* run folders here: 15
   wine_bamic_frozen_ep10_heads1_lamB0p1_lamS0p01_len100
   wine_bamic_frozen_ep15_heads1_lamB0p1_lamS0p01_len100
   wine_bamic_gatefix_rate0p075to0p125_posenc_selnorm_gatevalid_bestval_frozen_ep15_heads1_lamB0p1_lamS0p01_len100_seed20260526_ms1
   wine_bamic_gatefix_rate0p075to0p125_posenc_selnorm_gatevalid_bestval_frozen_ep15_heads1_lamB0p1_lamS0p01_len100_seed20260526_v1
   wine_bamic_gatefix_rate0p075to0p125_posenc_selnorm_gatevalid_bestval_frozen_ep15_heads1_lamB0p1_lamS0p01_len100_seed20260527_ms1
   wine_bamic_gatefix_rate0p075to0p125_posenc_selnorm_gatevalid_bestval_frozen_ep15_heads1_lamB0p1_lamS0p01_len100_seed20260528_ms1
   wine_bamic_gatefix_rate0p075to0p125_posenc_selnorm_gatevalid_bestval_frozen_ep15_heads1_lamB0p1_lamS0p01_len10

In [2]:
# ================================================================
# 1. Configuration
# ================================================================
import json
import numpy as np
import pandas as pd

RUN_LABEL = 'ms1'
GLOB = f'wine_bamic_*seed*_{RUN_LABEL}'
RESULTS_DIR = OUTPUTS_ROOT / f'multiseed_results_{RUN_LABEL}'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FROZEN_V1_GLOB = 'wine_bamic_*seed20260526_v1'   # paper run, for the cross-check
METRIC_COLS = ['acc', 'f1', 'auc', 'brier', 'nll', 'ece']

In [3]:
# ================================================================
# 2. Discover runs and check alignment
# ================================================================
REQUIRED = ['final_metrics.csv', 'test_document_uncertainty.csv',
            'example_word_uncertainty.csv', 'run_config.csv']

def read_seed(folder):
    metrics = pd.read_csv(folder / 'final_metrics.csv')
    seed = int(metrics['run_seed'].iloc[0])
    r = {'seed': seed, 'folder': folder, 'metrics': metrics,
         'docs': pd.read_csv(folder / 'test_document_uncertainty.csv'),
         'words': pd.read_csv(folder / 'example_word_uncertainty.csv')
                    .sort_values('position').reset_index(drop=True),
         'config': pd.read_csv(folder / 'run_config.csv').iloc[0]}
    p = folder / 'test_prob_samples.npy'
    r['prob_samples'] = np.load(p) if p.exists() else None          # [S, N]
    p = folder / 'panel_word_uncertainty.csv'
    r['panel'] = pd.read_csv(p) if p.exists() else None
    p = folder / 'param_posterior_sd.csv'
    r['param_sd'] = pd.read_csv(p) if p.exists() else None
    return r

folders = sorted(p for p in OUTPUTS_ROOT.glob(GLOB) if p.is_dir())
folders = [f for f in folders if all((f / x).exists() for x in REQUIRED)]
runs = [read_seed(f) for f in folders]
runs.sort(key=lambda r: r['seed'])
seeds = [r['seed'] for r in runs]
assert len(set(seeds)) == len(seeds), f'duplicate seeds: {seeds}'
print('seeds:', seeds)

n_docs = {len(r['docs']) for r in runs}
assert len(n_docs) == 1, f'test sizes differ: {n_docs}'
y_ref = runs[0]['docs']['y_true'].to_numpy()
for r in runs[1:]:
    assert np.array_equal(y_ref, r['docs']['y_true'].to_numpy()), \
        f"test split/order differs for seed {r['seed']}"
w_ref = runs[0]['words']['word'].tolist()
for r in runs[1:]:
    assert r['words']['word'].tolist() == w_ref, \
        f"example tokens differ for seed {r['seed']}"
print(f'[ok] {len(runs)} seeds, {len(y_ref)} shared test docs, '
      f'{len(w_ref)} example tokens')
print('draws present:', sum(r['prob_samples'] is not None for r in runs),
      '| panels present:', sum(r['panel'] is not None for r in runs))

seeds: [20260526, 20260527, 20260528, 20260529, 20260530, 20260531, 20260532, 20260533, 20260534, 20260535]
[ok] 10 seeds, 10000 shared test docs, 32 example tokens
draws present: 10 | panels present: 10


In [4]:
# ================================================================
# 3. Predictive metrics across seeds  (compare: Codex table section 2)
# ================================================================
rows = []
for r in runs:
    t = r['metrics'][r['metrics']['split'] == 'test'].iloc[0]
    rows.append({'seed': r['seed'],
                 **{c: float(t[c]) for c in METRIC_COLS},
                 'best_epoch': int(t['best_epoch']),
                 'tuned_tau_acc': float(r['config']['best_acc_threshold']),
                 'mean_gate_rate': float(r['docs']['selection_rate_mean'].mean()),
                 'mean_ci_width': float(r['docs']['ci_width'].mean()),
                 'median_ci_width': float(r['docs']['ci_width'].median())})
seed_table = pd.DataFrame(rows)
seed_table.to_csv(RESULTS_DIR / 'seed_level_metrics.csv', index=False)
print(seed_table.to_string(index=False))

summ_rows = []
for c in [c for c in seed_table.columns if c != 'seed']:
    v = seed_table[c].to_numpy(dtype=float)
    summ_rows.append({'metric': c, 'mean': v.mean(), 'sd': v.std(ddof=1),
                      'min': v.min(), 'max': v.max()})
metrics_summary = pd.DataFrame(summ_rows)
metrics_summary.to_csv(RESULTS_DIR / 'metrics_summary.csv', index=False)
print()
print(metrics_summary.to_string(index=False))

    seed    acc       f1      auc    brier      nll      ece  best_epoch  tuned_tau_acc  mean_gate_rate  mean_ci_width  median_ci_width
20260526 0.8749 0.824273 0.950469 0.086924 0.275693 0.027788           9          0.530        0.091584       0.031619         0.014376
20260527 0.8762 0.822484 0.948885 0.086820 0.275999 0.019249          10          0.545        0.090317       0.033094         0.015803
20260528 0.8710 0.822412 0.949739 0.088520 0.280066 0.040089           7          0.635        0.099999       0.038850         0.023651
20260529 0.8782 0.821303 0.949386 0.085524 0.271880 0.007682           7          0.460        0.084854       0.037742         0.022591
20260530 0.8737 0.822138 0.948780 0.087393 0.277029 0.025238           7          0.585        0.094757       0.036869         0.022571
20260531 0.8769 0.826937 0.950587 0.086131 0.273249 0.028255           8          0.565        0.094574       0.036479         0.020738
20260532 0.8775 0.819985 0.947360 0.087640 0.277

In [5]:
# ================================================================
# 4. Interval-routing stability  (compare: Codex routing table)
# Rule: auto-classify a document when its 95% credible interval
# excludes the threshold tau; defer otherwise.
# ================================================================
def routing_row(docs, tau, label_rule_tau):
    lo, hi = docs['ci95_lo'].to_numpy(), docs['ci95_hi'].to_numpy()
    p, y = docs['p_mean'].to_numpy(), docs['y_true'].to_numpy()
    auto = (lo > tau) | (hi < tau)
    pred = (p >= label_rule_tau).astype(int)
    err = pred != y
    cov = auto.mean()
    acc_auto = (pred[auto] == y[auto]).mean() if auto.any() else np.nan
    cap = err[~auto].sum() / max(err.sum(), 1)
    return cov, acc_auto, cap

rows = []
for r in runs:
    tau_t = float(r['config']['best_acc_threshold'])
    for name, tau, lt in [('fixed_0.5', 0.5, 0.5), ('tuned_acc', tau_t, tau_t)]:
        cov, acc_auto, cap = routing_row(r['docs'], tau, lt)
        rows.append({'seed': r['seed'], 'rule': name, 'tau': tau,
                     'auto_coverage': cov, 'acc_auto_classified': acc_auto,
                     'error_capture_deferred': cap})
routing = pd.DataFrame(rows)
routing.to_csv(RESULTS_DIR / 'routing_by_seed.csv', index=False)
routing_summary = (routing.groupby('rule')[
    ['auto_coverage', 'acc_auto_classified', 'error_capture_deferred']]
    .agg(['mean', 'std']))
print(routing_summary.to_string())
routing_summary.to_csv(RESULTS_DIR / 'routing_summary.csv')

          auto_coverage           acc_auto_classified           error_capture_deferred          
                   mean       std                mean       std                   mean       std
rule                                                                                            
fixed_0.5       0.96360  0.003499            0.888639  0.002066               0.140972  0.013676
tuned_acc       0.96336  0.003579            0.890452  0.003233               0.145923  0.014322


In [6]:
# ================================================================
# 5. Law-of-total-variance decomposition — document level
# Exact version from saved draws (probability AND logit scales);
# column-based fallback when draws are absent.
# Convention: within-fit variance over 100 MC draws (ddof=0),
# between-seed variance over seeds (ddof=1, sample variance).
# ================================================================
def decomposition(p_mean_by_seed, within_var_by_seed):
    within = within_var_by_seed.mean(axis=1)              # E_seed Var(Z|seed)
    between = p_mean_by_seed.var(axis=1, ddof=1)          # Var_seed E(Z|seed)
    total = within + between
    share = np.divide(between, total, out=np.zeros_like(total), where=total > 0)
    ratio = np.divide(between, within,
                      out=np.full_like(between, np.nan), where=within > 0)
    return pd.DataFrame({'within': within, 'between': between, 'total': total,
                         'between_share': share, 'between_within_ratio': ratio,
                         'between_exceeds_within': between > within})

def summarize(dec, name):
    d = dec
    return {'quantity': name,
            'mean_within': d['within'].mean(),
            'mean_between': d['between'].mean(),
            'ratio_of_means': d['between'].mean() / d['within'].mean(),
            'between_share_of_means': d['between'].mean() / d['total'].mean(),
            'median_between_share': d['between_share'].median(),
            'pct_between_exceeds_within': 100 * d['between_exceeds_within'].mean()}

summaries, doc_tables = [], {}

# --- from summary columns (always available) ---
P = np.column_stack([r['docs']['p_mean'].to_numpy() for r in runs])
V = np.column_stack([r['docs']['var'].to_numpy() for r in runs])
dec_p_cols = decomposition(P, V)
doc_tables['prob_from_columns'] = dec_p_cols
summaries.append(summarize(dec_p_cols, 'doc probability (summary columns)'))

# --- exact, from saved draws ---
have_draws = [r for r in runs if r['prob_samples'] is not None]
if len(have_draws) == len(runs):
    Pm, Vm, Lm, Lv = [], [], [], []
    for r in runs:
        S = r['prob_samples'].astype(np.float64)          # [S, N]
        Pm.append(S.mean(axis=0)); Vm.append(S.var(axis=0))
        eps = 1e-7
        L = np.log(np.clip(S, eps, 1 - eps) / np.clip(1 - S, eps, 1 - eps))
        Lm.append(L.mean(axis=0)); Lv.append(L.var(axis=0))
    dec_p = decomposition(np.column_stack(Pm), np.column_stack(Vm))
    dec_l = decomposition(np.column_stack(Lm), np.column_stack(Lv))
    doc_tables['prob_from_draws'] = dec_p
    doc_tables['logit_from_draws'] = dec_l
    summaries.append(summarize(dec_p, 'doc probability (exact draws)'))
    summaries.append(summarize(dec_l, 'doc logit (exact draws)'))
    # label stability across seeds at 0.5
    lab = (np.column_stack(Pm) >= 0.5)
    unanimous = (lab == lab[:, [0]]).all(axis=1)
    print(f'labels unanimous across seeds: {100 * unanimous.mean():.2f}% of docs')
else:
    print(f'[note] draws present for only {len(have_draws)}/{len(runs)} seeds; '
          f'exact decomposition skipped (column-based version above stands).')

for name, tbl in doc_tables.items():
    tbl.to_csv(RESULTS_DIR / f'document_decomposition_{name}.csv', index=False)
doc_summary = pd.DataFrame(summaries)
doc_summary.to_csv(RESULTS_DIR / 'document_decomposition_summary.csv', index=False)
print()
print(doc_summary.to_string(index=False))

labels unanimous across seeds: 90.50% of docs

                         quantity  mean_within  mean_between  ratio_of_means  between_share_of_means  median_between_share  pct_between_exceeds_within
doc probability (summary columns)     0.000178      0.002019       11.321734                0.918843              0.894852                       99.95
    doc probability (exact draws)     0.000177      0.002019       11.399372                0.919351              0.895728                       99.98
          doc logit (exact draws)     0.013275      0.152313       11.473997                0.919833              0.904337                       99.98


In [7]:
# ================================================================
# 6. Token-level decomposition — 200-document panel + example sentence
# ================================================================
token_summary = None
panels = [r for r in runs if r['panel'] is not None]
if len(panels) == len(runs):
    keys = ['doc_index', 'position']
    base = runs[0]['panel'][keys + ['word']].copy()
    for r in runs:
        assert len(r['panel']) == len(base), 'panel row counts differ'
        assert (r['panel']['word'].to_numpy() == base['word'].to_numpy()).all(), \
            f"panel tokens differ for seed {r['seed']}"
    Zm = np.column_stack([r['panel']['z_mean'].to_numpy() for r in runs])
    Zv = np.column_stack([r['panel']['z_sd'].to_numpy() ** 2 for r in runs])
    Dm = np.column_stack([r['panel']['delta_mean'].to_numpy() for r in runs])
    dec_tok = decomposition(Zm, Zv)
    dec_tok = pd.concat([base.reset_index(drop=True), dec_tok], axis=1)
    dec_tok['z_mean_avg'] = Zm.mean(axis=1)
    dec_tok['delta_mean_avg'] = Dm.mean(axis=1)
    dec_tok.to_csv(RESULTS_DIR / 'token_decomposition_panel.csv', index=False)
    token_summary = summarize(dec_tok, 'token contribution (panel)')
    sel = dec_tok[dec_tok['delta_mean_avg'] >= 0.5]
    token_sel_summary = summarize(sel, 'token contribution (avg-selected tokens)')
    print(pd.DataFrame([token_summary, token_sel_summary]).to_string(index=False))
    print(f'panel: {len(dec_tok)} token positions, '
          f'{len(sel)} with average gate >= 0.5')
else:
    print(f'[note] panel present for only {len(panels)}/{len(runs)} seeds.')

# Example sentence (32 tokens of the Section-5.7 review) — always available
Zm = np.column_stack([r['words']['z_mean'].to_numpy() for r in runs])
Zv = np.column_stack([r['words']['z_sd'].to_numpy() ** 2 for r in runs])
Pp = np.column_stack([r['words']['p_z_positive'].to_numpy() for r in runs])
dec_ex = decomposition(Zm, Zv)
dec_ex.insert(0, 'word', runs[0]['words']['word'].to_numpy())
dec_ex.insert(0, 'position', runs[0]['words']['position'].to_numpy())
dec_ex['z_mean_avg'] = Zm.mean(axis=1)
modal_sign = np.sign(np.median(Zm, axis=1))
dec_ex['sign_agreement'] = (np.sign(Zm) == modal_sign[:, None]).mean(axis=1)
dec_ex['p_z_positive_min'] = Pp.min(axis=1)
dec_ex['p_z_positive_max'] = Pp.max(axis=1)
dec_ex = dec_ex.sort_values('z_mean_avg', key=np.abs, ascending=False)
dec_ex.to_csv(RESULTS_DIR / 'example_word_stability.csv', index=False)
print()
print(dec_ex.head(10).to_string(index=False))

                                quantity  mean_within  mean_between  ratio_of_means  between_share_of_means  median_between_share  pct_between_exceeds_within
              token contribution (panel)     0.000032      0.003788      118.546492                0.991635              0.984597                     99.9544
token contribution (avg-selected tokens)          NaN           NaN             NaN                     NaN                   NaN                         NaN
panel: 6579 token positions, 0 with average gate >= 0.5

 position       word   within  between    total  between_share  between_within_ratio  between_exceeds_within  z_mean_avg  sign_agreement  p_z_positive_min  p_z_positive_max
        7     floral 0.000115 0.010256 0.010371       0.988892             89.026099                    True    0.310134             1.0              1.00               1.0
       17      stays 0.000078 0.007474 0.007552       0.989721             96.286002                    True   -0.192839   

In [8]:
# ================================================================
# 7. Seed-ensemble predictive (constructive output)
# Pooling draws across seeds approximates the TOTAL-uncertainty
# predictive that Dr. Cao's decomposition points at: the pooled
# variance equals within + between by construction.
# ================================================================
ensemble = None
if all(r['prob_samples'] is not None for r in runs):
    pooled = np.concatenate([r['prob_samples'] for r in runs], axis=0)  # [10*S, N]
    y = runs[0]['docs']['y_true'].to_numpy()
    p_mean = pooled.mean(axis=0)
    lo = np.quantile(pooled, 0.025, axis=0)
    hi = np.quantile(pooled, 0.975, axis=0)
    from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                                 brier_score_loss, log_loss)
    pred = (p_mean >= 0.5).astype(int)
    auto = (lo > 0.5) | (hi < 0.5)
    err = pred != y
    ensemble = {
        'n_pooled_draws': int(pooled.shape[0]),
        'acc': accuracy_score(y, pred),
        'f1': f1_score(y, pred, zero_division=0),
        'auc': roc_auc_score(y, p_mean),
        'brier': brier_score_loss(y, p_mean),
        'nll': log_loss(y, np.clip(p_mean, 1e-7, 1 - 1e-7), labels=[0, 1]),
        'mean_ci_width_ensemble': float((hi - lo).mean()),
        'median_ci_width_ensemble': float(np.median(hi - lo)),
        'mean_ci_width_single_fit': float(np.mean(
            [r['docs']['ci_width'].mean() for r in runs])),
        'auto_coverage_tau0.5': float(auto.mean()),
        'acc_auto_classified': float((pred[auto] == y[auto]).mean()),
        'error_capture_deferred': float(err[~auto].sum() / max(err.sum(), 1)),
    }
    pd.DataFrame([ensemble]).to_csv(RESULTS_DIR / 'seed_ensemble_summary.csv',
                                    index=False)
    for k, v in ensemble.items():
        print(f'{k:28s}: {v:.4f}' if isinstance(v, float) else f'{k:28s}: {v}')
else:
    print('[note] ensemble skipped — draws missing for some seeds.')

n_pooled_draws              : 1000
acc                         : 0.8770
f1                          : 0.8259
auc                         : 0.9509
brier                       : 0.0854
nll                         : 0.2710
mean_ci_width_ensemble      : 0.1112
median_ci_width_ensemble    : 0.0636
mean_ci_width_single_fit    : 0.0351
auto_coverage_tau0.5        : 0.8896
acc_auto_classified         : 0.9157
error_capture_deferred      : 0.3902


In [9]:
# ================================================================
# 8. Variational posterior SDs vs initialization
# init_rho = -5.0  =>  softplus(-5) ~ 0.00672. Posterior SDs that sit
# at ~0.0067 have not moved from initialization — important context
# for interpreting narrow within-fit intervals and any prior-
# sensitivity null result.
# ================================================================
ps = [r for r in runs if r['param_sd'] is not None]
if ps:
    tab = pd.concat([r['param_sd'].assign(seed=r['seed']) for r in ps])
    agg = tab.groupby('module')[['mean_posterior_sd', 'median_posterior_sd',
                                 'max_posterior_sd']].agg(['mean', 'std'])
    print(agg.to_string())
    print(f"\ninit value softplus(-5) = {np.log1p(np.exp(-5.0)):.5f}")
    tab.to_csv(RESULTS_DIR / 'param_posterior_sd_by_seed.csv', index=False)
else:
    print('[note] param_posterior_sd.csv not found in any run.')

           mean_posterior_sd           median_posterior_sd           max_posterior_sd          
                        mean       std                mean       std             mean       std
module                                                                                         
b_layer             0.007556  0.000336            0.007477  0.000325         0.009317  0.001026
beta_layer          0.007030  0.000116            0.007008  0.000102         0.007587  0.000300

init value softplus(-5) = 0.00672


In [10]:
# ================================================================
# 9. Same-seed reproducibility: 20260526_ms1 vs frozen paper run (_v1)
# ================================================================
frozen = sorted(OUTPUTS_ROOT.glob(FROZEN_V1_GLOB))
ms1_26 = next((r for r in runs if r['seed'] == 20260526), None)
if frozen and ms1_26 is not None:
    fdocs = pd.read_csv(frozen[0] / 'test_document_uncertainty.csv')
    fmet = pd.read_csv(frozen[0] / 'final_metrics.csv')
    ft = fmet[fmet['split'] == 'test'].iloc[0]
    mt = ms1_26['metrics'][ms1_26['metrics']['split'] == 'test'].iloc[0]
    print('same-seed check (CUDA nondeterminism means not bit-identical):')
    for c in METRIC_COLS:
        print(f'  {c:6s}: frozen_v1={float(ft[c]):.4f}  ms1={float(mt[c]):.4f}')
    dp = (fdocs['p_mean'].to_numpy() - ms1_26['docs']['p_mean'].to_numpy())
    print(f'  p_mean abs diff: mean={np.abs(dp).mean():.4f}, '
          f'max={np.abs(dp).max():.4f}')
else:
    print('[note] frozen _v1 folder or ms1 seed 20260526 not found; skipped.')

same-seed check (CUDA nondeterminism means not bit-identical):
  acc   : frozen_v1=0.8749  ms1=0.8749
  f1    : frozen_v1=0.8243  ms1=0.8243
  auc   : frozen_v1=0.9505  ms1=0.9505
  brier : frozen_v1=0.0869  ms1=0.0869
  nll   : frozen_v1=0.2757  ms1=0.2757
  ece   : frozen_v1=0.0278  ms1=0.0278
  p_mean abs diff: mean=0.0000, max=0.0000


In [11]:
# ================================================================
# 10. Side-by-side with Codex's reported numbers (its independent
# reimplementation, MPS backend). Large disagreements = investigate.
# ================================================================
codex = {
    'source': 'Codex report 2026-08-17 (reimplemented pipeline, M2 Ultra/MPS)',
    'acc_mean': 0.874730, 'acc_sd': 0.003073,
    'f1_mean': 0.823164, 'auc_mean': 0.949137, 'nll_mean': 0.278315,
    'ece_mean': 0.028366, 'val_gate_rate_mean': 0.094300,
    'doc_prob_within': 1.8661e-4, 'doc_prob_between': 2.1603e-3,
    'doc_prob_ratio': 11.58, 'doc_prob_between_share': 0.920,
    'doc_logit_within': 1.4288e-2, 'doc_logit_between': 1.8041e-1,
    'token_within': 3.0398e-5, 'token_between': 3.5123e-3,
    'token_ratio': 115.54, 'token_between_share': 0.991,
    'pct_between_exceeds_within_docs': 99.96,
    'routing_coverage': 0.96336, 'routing_acc_auto': 0.88789,
    'routing_error_capture': 0.13781,
}
ms = metrics_summary.set_index('metric')
ours = {
    'acc_mean': ms.loc['acc', 'mean'], 'acc_sd': ms.loc['acc', 'sd'],
    'f1_mean': ms.loc['f1', 'mean'], 'auc_mean': ms.loc['auc', 'mean'],
    'nll_mean': ms.loc['nll', 'mean'], 'ece_mean': ms.loc['ece', 'mean'],
}
dsum = doc_summary.set_index('quantity')
key = ('doc probability (exact draws)'
       if 'doc probability (exact draws)' in dsum.index
       else 'doc probability (summary columns)')
ours.update({
    'doc_prob_within': dsum.loc[key, 'mean_within'],
    'doc_prob_between': dsum.loc[key, 'mean_between'],
    'doc_prob_ratio': dsum.loc[key, 'ratio_of_means'],
    'doc_prob_between_share': dsum.loc[key, 'between_share_of_means'],
    'pct_between_exceeds_within_docs': dsum.loc[key, 'pct_between_exceeds_within'],
})
if 'doc logit (exact draws)' in dsum.index:
    ours['doc_logit_within'] = dsum.loc['doc logit (exact draws)', 'mean_within']
    ours['doc_logit_between'] = dsum.loc['doc logit (exact draws)', 'mean_between']
rt = routing[routing['rule'] == 'tuned_acc']
ours.update({'routing_coverage': rt['auto_coverage'].mean(),
             'routing_acc_auto': rt['acc_auto_classified'].mean(),
             'routing_error_capture': rt['error_capture_deferred'].mean()})
if token_summary is not None:
    ours.update({'token_within': token_summary['mean_within'],
                 'token_between': token_summary['mean_between'],
                 'token_ratio': token_summary['ratio_of_means'],
                 'token_between_share': token_summary['between_share_of_means']})
comp = pd.DataFrame({'codex_reported': pd.Series(
                        {k: v for k, v in codex.items() if k != 'source'}),
                     'ours_official_pipeline': pd.Series(ours)})
comp['rel_diff'] = ((comp['ours_official_pipeline'] - comp['codex_reported'])
                    / comp['codex_reported'])
comp.to_csv(RESULTS_DIR / 'codex_vs_ours.csv')
print(codex['source'])
print(comp.to_string())

Codex report 2026-08-17 (reimplemented pipeline, M2 Ultra/MPS)
                                 codex_reported  ours_official_pipeline  rel_diff
acc_mean                               0.874730                0.875070  0.000389
acc_sd                                 0.003073                0.002266 -0.262704
auc_mean                               0.949137                0.949430  0.000309
doc_logit_between                      0.180410                0.152313 -0.155738
doc_logit_within                       0.014288                0.013275 -0.070924
doc_prob_between                       0.002160                0.002019 -0.065632
doc_prob_between_share                 0.920000                0.919351 -0.000706
doc_prob_ratio                        11.580000               11.399372 -0.015598
doc_prob_within                        0.000187                0.000177 -0.051110
ece_mean                               0.028366                0.025821 -0.089704
f1_mean                            

In [12]:
# ================================================================
# 11. Save the JSON summary for the joint write-up
# ================================================================
def jsonable(x):
    if isinstance(x, (np.floating, np.integer)):
        return x.item()
    return x

summary = {
    'seeds': seeds,
    'n_test_docs': int(len(y_ref)),
    'metrics_summary': metrics_summary.to_dict(orient='records'),
    'routing_summary': routing.groupby('rule').mean(numeric_only=True)
                              .reset_index().to_dict(orient='records'),
    'document_decomposition': doc_summary.to_dict(orient='records'),
    'token_panel_decomposition': token_summary,
    'seed_ensemble': ensemble,
    'notes': ('within-fit = MC var over 100 draws (ddof=0); '
              'between-seed = sample var over seeds (ddof=1); '
              'ratio_of_means = mean(between)/mean(within)'),
}
with open(RESULTS_DIR / 'multiseed_claude_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=jsonable)
print('written:', RESULTS_DIR / 'multiseed_claude_summary.json')
print('\nSend back to Claude: multiseed_claude_summary.json, '
      'metrics_summary.csv, document_decomposition_summary.csv, '
      'token_decomposition_panel.csv (or its summary), '
      'seed_ensemble_summary.csv, codex_vs_ours.csv')

written: /content/drive/MyDrive/AMIC project/wine_benchmark/outputs/multiseed_results_ms1/multiseed_claude_summary.json

Send back to Claude: multiseed_claude_summary.json, metrics_summary.csv, document_decomposition_summary.csv, token_decomposition_panel.csv (or its summary), seed_ensemble_summary.csv, codex_vs_ours.csv
